# Introduction
This notebook clones and runs this repo (V7 branch): https://github.com/beetree/ARC-AGI

For this notebook to run properly you have to run it in "batch mode". Click "Save Version", choose "Save & Run All", then click "Save". This will run the notebook in the background.

You can follow-along on the progress in the notebook as it outputs its progress to STDOUT. It will look like this:

```
Legend: ⚡ Running   ⏳ Queued   ✅ Done

| Status         | Task:Test  | Step  | Phase           | Time     | Message
|----------------|------------|-------|-----------------|----------|---------------------
|  ⚡5 ⏳0 ✅5  | 13e47133:1 |  5/5  | Full search     | 10711.2s | Going DEEP: 0/2/1/0 left
|  ⚡5 ⏳0 ✅5  | f560132c:1 |  5/5  | Full search     | 11200.1s | Going DEEP: 0/0/1/0 left
|  ⚡5 ⏳0 ✅5  | 16b78196:1 |  5/5  | Full search     | 11473.3s | Going DEEP: 0/0/1/0 left
|  ⚡5 ⏳0 ✅5  | 13e47133:1 |  5/5  | Full search     | 11790.1s | Going DEEP: 0/1/1/0 left
|  ⚡5 ⏳0 ✅5  | e87109e9:1 |  DONE | Finished        | 11731.3s | PASS ($31.3485)
|  ⚡4 ⏳0 ✅6  | 13e47133:1 |  5/5  | Full search     | 11952.1s | Going DEEP: 0/0/1/0 left
|  ⚡4 ⏳0 ✅6  | f560132c:1 |  DONE | Finished        | 12305.6s | PASS ($24.7502)
```

You can track the progress of the overall job by looking at the "Status" column which tracks the status of all task:test pairs (~167 in the arc agi 2 eval data set).

You need to set Kaggle secrets for the repo to run. To do this, create a single secret "CONFIG" that has the following value (comma separated, just copy paste it, it works):

```
OPENAI_API_KEY={insert key},CLAUDE_API_KEY={insert key},GEMINI_API_KEY={insert key}
```

IMPORTANT: Kaggle's output logging for batch modes sometimes look buggy, and it can seem like the notebook is "restarting" as it sometimes ends up "replaying" all the STDOUT. This is purely a web-output-streaming bug and can safely be ignored. Just look at the very bottom of the log and you'll see the current state.

IMPORTANT: Sometimes some LLM calls fail. This will output in a noisy way to STDOUT, including the retry logic. This is fine, just let it keep running. Even if all the retries fail, the solver is robust enough to anyway come to a reasonable solution.

# Install dependencies

In [ ]:
!echo "Starting ARC-AGI notebook, $(date '+%Y-%m-%d %H:%M:%S')"


print("Checking out repo:")

!git clone -b Johan_Land_Solver_V7 https://github.com/beetree/ARC-AGI
%cd ARC-AGI

print("\n\n-------------------------------\nCurrent version / last commit on repo:")
!git -c advice.detachedHead=false checkout 2f77746dcfe4da6bc689ce47db809b8b5339613b
print("-------------------------------\n")


!pip install -r requirements-kaggle.txt -q --no-warn-conflicts

print("\nInstalled all dependencies")

# Set up API KEYs

Create a Kaggle notebook Secret by clicking on "Add-ons" in the menu, then "Secrets"
Click on "Add Secret" in the bottom and add these secrets

```
OPENAI_API_KEY=...
CLAUDE_API_KEY=...
GEMINI_API_KEY=...
```


In [ ]:

import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

os.environ["CLAUDE_API_KEY"] = user_secrets.get_secret("CLAUDE_API_KEY")
os.environ["GEMINI_API_KEY"] = user_secrets.get_secret("GEMINI_API_KEY")
os.environ["OPENAI_API_KEY"] = user_secrets.get_secret("OPENAI_API_KEY")



# Get test dataset

In [ ]:
!git clone https://github.com/arcprize/ARC-AGI-2.git
!mv ARC-AGI-2/data/evaluation .
!rm -rf ARC-AGI-2

file_path = "/kaggle/input/dataset/arc-agi_evaluation_challenges.json"



# Run the solver

In [ ]:
from src.runner import run_app

if __name__ == "__main__":
    run_app(task_file=file_path, task_workers=85)
    #run_app(task_file=file_path, task_workers=85, solver_testing=True, task_limit=1)


# Extract logs and submission

The log contains a file `submission.tgz` which has a directory `submissions/` inside of it.

To score the submissions, just clone `https://github.com/arcprize/arc-agi-benchmarking` and then run a command like this:

`python src/arc_agi_benchmarking/scoring/scoring.py --task_dir the_tasks_with_solutions/ --submission_dir submissions/ --results_dir results/`

The solver can also score itself and produce a results.json file, but it can only do so if the original tasks include solutions. If there are no solutions in the original input tasks, only the answers will be produced, and you'll need to score the solutions separately, as per above.


In [ ]:
if __name__ == "__main__":
    print("\n\n")
    print("Script completed successfully")
    print("packaging up logs and submission file in base directory")
    !tar -zcf logs.tgz logs/
    !mv logs.tgz ../
    !mv submissions/submission.json ../
    !tar -zcf submissions.tgz submissions/
    !mv submissions.tgz ../
    
    
